## inference for variance-ratio

#### 모듈

In [20]:
# _rd_00.py
# 분산 추론. 해보자.
# 분산비 추론.
# F비
# raw data - 연봉, 경력 기간, 성별.
# _rd_00.py ==> 이건 벤치마크용.


import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp, f


pd.options.display.float_format = '{:.4f}'.format
np.set_printoptions(suppress=True, precision=4)


### 분산비 검정 함수
분산비=1을 검정함. 즉, 등분산 검정.

In [21]:
def variance_ratio_test(x, y, alpha) : # 등분산 검정임
    x = np.asarray(x)  # 이게 버릇이군. 이하 ndarray로 변환/유지한다는 뜻. 일관성 유지.
    y = np.asarray(y)

    n1 = len(x)
    n2 = len(y)

    s1 = np.var(x, ddof=1)   # 자유도를 일상적으로 지정하자. 디폴트로 넘어가다가 미세 차이 발생 가능.
    s2 = np.var(y, ddof=1)

    F = s1 / s2              # 이것이 phi_hat과 같음. 등분산 가설을 검정하는 통계량임.
    df1 = n1 - 1
    df2 = n2 - 1
                             # 분포 호출 시, cdf, sf, ppf, isf 참조. 1-cdf보다는 sf.
    p = 2 * min( f.cdf(F, df1, df2),
                 f.sf(F, df1, df2) )
              #  1 - f.cdf(F, df1, df2))  # 이것보다 f.sf를 쓰라는, 그게 안정적이라는.

    ci = (
        F / f.ppf(1 - alpha/2, df1, df2 ),
        F / f.ppf(  alpha/2, df1, df2 )
        )

    return F, p, ci

### 데이터 읽기, 원격

In [22]:
# 1. 데이터 준비. 읽기. 생성.
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url)
df_dat.head()

gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
hgt = df_dat['ht'].to_numpy()
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환.

df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임
hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

n_all = len(df_dat)
n_m = len(hgt_m)
n_f = len(hgt_f)

### F-ratio, F비, 분산비, 분포
\begin{align}
F & = {\hat\phi \over \phi }  \\
     & \sim F_{(df_1, df_2 )}
\end{align}

### 신뢰구간
$ 100{(1-\alpha) } \% $ 신뢰구간
\begin{align}
 {\hat \phi \over f_2 } < \phi <  {\hat \phi \over f_1 }
\end{align}
$ P( F > f_2 ) = P( F < f_1 ) = {\alpha \over 2 } $

### 유의(신뢰)수준, 자유도

In [23]:

# 분포 임계치, 양방향, 우측값. 적당한 위치 모색.
# 분산은 카이제곱 분포. 분산비는 F분포.
# 임계치, 전체
alpha = 0.05                          # significance level, two side.
#    kcv_r = stats.chi2.ppf(1-alpha/2, df_all ) # critical value on the right, two-side
#    kcv_l = stats.chi2.ppf(alpha/2, df_all ) # critical value on the left,
#    kcv_r1 = stats.chi2.ppf(1-alpha, df_all ) # critical value on the right, one-side
#    zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
#    tcv_r = stats.t.ppf(1 - alpha / 2, df= n_tall)

# 3. 분산비: 성별 대비. male/female
# 추정치: 분산
# overall, 불요, 그룹별( m - f) 비교임
alpha, 100*(1-alpha), n_m-1, n_f-1

(0.05, 95.0, 8939, 11187)

### 표본 분산비, 자유도, 신뢰구간

In [24]:

sig2hat_m = hgt_m.var(ddof=1)  # ddof=1 <-- 표본분산; ddof=0 <-- 모분산, mle.
dof_m = n_m -1

sig2hat_f = hgt_f.var(ddof=1)
dof_f = n_f -1

# 표본 분산비 male/female
phi_hat = sig2hat_m / sig2hat_f

# 임계치, 양쪽, 2개, F분포
f_2 = stats.f.ppf( 1 - alpha/2, dof_m, dof_f)
f_1 = stats.f.ppf( alpha/2, dof_m, dof_f)

# 신뢰구간, right, left,
ci_r = phi_hat / f_1
ci_l = phi_hat / f_2
print("모 분산비 추론: ")
print( "     F ratio,      confidence interval ")
print( phi_hat,  ci_l, ci_r )



모 분산비 추론: 
     F ratio,      confidence interval 
1.2232853926694627 1.1761651070365704 1.2724048384610227


### 가설 설정
\begin{align}
H_0 &: \phi = \phi_0 \\
H_A &: \phi \ne \phi_0
\end{align}

### 검정통계량, 기각/채택
\begin{align}
F_0 & =  { \phi \over  \phi_0 }
\end{align}
reject $ H_0 $ if  $ F_0 > f_2 $  or $ F_0 < f_1 $

In [25]:
# 가설검정. 전체, 톨
# 귀무가설 H0: mu = mu0
# 검정통계치, pvalue

phi_zero = 1   # 등분산.

print("H0: phihat = ", phi_zero, " HA: phi is not ", phi_zero )


H0: phihat =  1  HA: phi is not  1


### F 통계치, 검정결과, p값

In [26]:
# F statistics
f_0 =  phi_hat / phi_zero   # F 검정통계량, 이것은 엄밀하게 f_0 is not phi_hat.

yn_h0 = " 'reject h0' " if (f_0 > f_2 or f_0 < f_1 )  else " 'fail to reject h0' "   # 이건 되는 군.
res = int( f_0 > f_2  or f_0 < f_1 )   # 이것은 기각 성공 1, 기각 실패 0

a = stats.f.cdf( f_0, dof_m, dof_f )
b = stats.f.sf( f_0, dof_m, dof_f )    # 1- cdf , f.sf가 더 안정적이라는 거라고...
pval = 2 * min(a, b)

print(" H0: phi = ", phi_zero)
print("    F비 ,     임계치 (f_1, f_2) ,    검정결과,    p값 ")
print( f_0 , f_1, f_2, yn_h0, pval)


 H0: phi =  1
    F비 ,     임계치 (f_1, f_2) ,    검정결과,    p값 
1.2232853926694627 0.96139636984486 1.040062645415162  'reject h0'  7.106101391380385e-24


### 함수 이용 예시,
앞서 정의한 분산비검정함수 이용

In [27]:

# phi의 신뢰구간 산식 assisted by chatgpt   # 이것은 직접 계산한 것과 같음.
lower = phi_hat / f.ppf(1 - alpha/2, dof_m, dof_f )
upper = phi_hat / f.ppf( alpha/2, dof_m, dof_f )

# pval = 2 * min( f.cdf( f_0 , dof_m, dof_f),
#                  f.sf( f_0, dof_m, dof_f ) )
#                   1 - f.cdf(phi_hat , dof_m, dof_f))

print(" 표본 분산비(phi_hat), 모 분산비(phi)에 대한 95% CI ")
print( f"{ phi_hat:.4f} , ( { lower:.4f}, { upper:.4f} )." )

# 별도 셀에 정의된 함수가 돌려주는 것. F-stat, p-value, CI given alpha
F, p, c = variance_ratio_test( hgt_m, hgt_f, alpha )
print(" 등분산 검정 ")
print(" F ratio,  CI of phi  ")
print(  F ,   c  )



 표본 분산비(phi_hat), 모 분산비(phi)에 대한 95% CI 
1.2233 , ( 1.1762, 1.2724 ).
 등분산 검정 
 F ratio,  CI of phi  
1.2232853926694627 (np.float64(1.1761651070365704), np.float64(1.2724048384610227))


## 이 결과를 2개 그룹의 평균 비교에 적용함.
별도 주제로 검토함.